In [1]:
import json
import onnx
import torch
import numpy as np
import pandas as pd
from torch import nn

In [2]:
class ExampleModel(nn.Module):

    def __init__(self):
        super(ExampleModel, self).__init__()
        self.x_latent_dim = 12
        self.x_physical_dim = 3
        self.u_dim = 2
        self.y_dim = 1

        self.register_buffer('initialized', torch.tensor(0, dtype=torch.bool))

        # set random seed for reproducibility
        torch.manual_seed(42)

        self.dynamic = nn.Sequential(
            nn.Linear(self.x_latent_dim + self.u_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 3)
        )
        self.output = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

        self.init = nn.Sequential(
            nn.Linear(self.x_physical_dim, 16),
            nn.ReLU(),
            nn.Linear(16, self.x_latent_dim)
        )

    def forward(self, u, x_0_physical, x_0_latent):
        # initialize latent state if not already initialized
        # if self.initialized == 0:
        #     x_0_latent = self.initialize(x_0_physical, x_0_latent)
        # else:
        #     x_0_latent = self.no_function(x_0_physical, x_0_latent)

        def initialize(x_0_physical, x_0_latent):
            x_0_latent = self.init(x_0_physical)
            # self.initialized.fill_(1)
            return x_0_latent.clone()

        def no_function(x_0_physical, x_0_latent):
            return x_0_latent.clone()

        x_0_latent = torch.cond(self.initialized == torch.tensor(0, dtype=torch.bool),
                   initialize,
                   no_function,
                   (x_0_physical, x_0_latent)
                   )
        
        x_1 = self.dynamic(torch.cat((x_0_latent, u), dim=0))
        y_1 = self.output(x_1)
        return y_1, x_1

# Compile the model
# ExampleModel = torch.compile(ExampleModel)

In [3]:
# Create the model
model = ExampleModel()


# Create three tensors
u = torch.tensor([0.5, -0.2], dtype=torch.float32)  # Control input
x_0 = torch.tensor([1.0, 0.0, -1.0], dtype=torch.float32)  # Initial state
x_0_latent = torch.zeros(model.x_latent_dim, dtype=torch.float32)  # Initial latent state

# Run the model
print("Before forward pass, initialized:", model.initialized.item())
print("Condition before forward pass:", model.initialized == torch.tensor(0, dtype=torch.bool))
y_1, x_1 = model(u, x_0, x_0_latent)
print("After forward pass, initialized:", model.initialized.item())
print("Condition after forward pass:", model.initialized == torch.tensor(0, dtype=torch.bool))

print("Output y:", y_1[0].item())

Before forward pass, initialized: False
Condition before forward pass: tensor(True)
After forward pass, initialized: False
Condition after forward pass: tensor(True)
Output y: -0.08785043656826019


In [ ]:
model_name = "example6"
# Set the model to evaluation mode
model = ExampleModel()
model.eval()
import os
# set environment variable TORCHDYNAMO_VERBOSE=1 to see dynamo logs
os.environ["TORCHDYNAMO_VERBOSE"] = "1"
os.environ["TORCH_LOGS"] = "+dynamo"


# Save the model in ONNX format
torch.onnx.export(
    model,
    (u, x_0, x_0_latent),
    f"{model_name}.onnx",
    verbose=True,
    input_names=["u", "x_0_physical", "x_0_latent"],
    output_names=["ynext", "x_latent_next"],
    dynamo=True,
)

# # Load the model
# onnx_model = onnx.load(f"{model_name}.onnx")

# # Check the model
# onnx.checker.check_model(onnx_model)

# # Add description to the model
# onnx_model.graph.doc_string = "Example to test FMU with local variables."

# # Add metadata to the model
# onnx_model.producer_name = "ExampleModel"
# onnx_model.producer_version = "0.0.1"
# onnx_model.domain = "example"
# onnx_model.model_version = 1

# # Save the model
# onnx.save(onnx_model, f"{model_name}.onnx")


[torch.onnx] Obtain model graph for `ExampleModel([...]` with `torch.export.export(..., strict=False)`...


E1104 18:05:45.257000 15804 Lib\site-packages\torch\_guards.py:368] [0/1] Error while creating guard:
E1104 18:05:45.257000 15804 Lib\site-packages\torch\_guards.py:368] [0/1] Name: ''
E1104 18:05:45.257000 15804 Lib\site-packages\torch\_guards.py:368] [0/1]     Source: shape_env
E1104 18:05:45.257000 15804 Lib\site-packages\torch\_guards.py:368] [0/1]     Create Function: SHAPE_ENV
E1104 18:05:45.257000 15804 Lib\site-packages\torch\_guards.py:368] [0/1]     Guard Types: ['SHAPE_ENV']
E1104 18:05:45.257000 15804 Lib\site-packages\torch\_guards.py:368] [0/1]     Code List: ["0 <= L['args'][3][1].size()[0]"]
E1104 18:05:45.257000 15804 Lib\site-packages\torch\_guards.py:368] [0/1]     Object Weakref: None
E1104 18:05:45.257000 15804 Lib\site-packages\torch\_guards.py:368] [0/1]     Guarded Class Weakref: None
E1104 18:05:45.257000 15804 Lib\site-packages\torch\_guards.py:368] [0/1] Traceback (most recent call last):
E1104 18:05:45.257000 15804 Lib\site-packages\torch\_guards.py:368] [0/

[torch.onnx] Obtain model graph for `ExampleModel([...]` with `torch.export.export(..., strict=False)`... ❌
[torch.onnx] Obtain model graph for `ExampleModel([...]` with `torch.export.export(..., strict=True)`...
[torch.onnx] Obtain model graph for `ExampleModel([...]` with `torch.export.export(..., strict=True)`... ❌


TorchExportError: Failed to export the model with torch.export. [96mThis is step 1/3[0m of exporting the model to ONNX. Next steps:
- Modify the model code for `torch.export.export` to succeed. Refer to https://pytorch.org/docs/stable/generated/exportdb/index.html for more information.
- Debug `torch.export.export` and submit a PR to PyTorch.
- Create an issue in the PyTorch GitHub repository against the [96m*torch.export*[0m component and attach the full error stack as well as reproduction scripts.

## Exception summary

<class 'torch._dynamo.exc.InternalTorchDynamoError'>: RuntimeError: Compiler: cl is not found.


(Refer to the full stack trace above for more information.)

## Generating model description

Create and save the model description to be provided to ONNX2FMU.

In [12]:
model_description = {
    "name": "example5",
    "description": "Example to test FMU with local variables.",
    "FMIVersion": "2.0",
    "inputs": [
        {
            "name": "u",
            "description": "A vector of control variables at time t, size 2."
        },
    ],
    "outputs": [
        {
            "name": "ynext",
            "description": "The output variable at time t+1, size 1."
        }
    ],
    "locals": [
        {
            "nameIn": "x",
            "nameOut": "xnext",
            "description": "The local state variable, size 3.",
            "start": [
                0.0,
                0.0,
                0.0
            ]
        },
    ]
}

# Save model description
with open(f"{model_name}Description.json", "w", encoding="utf-8") as f:
    json.dump(model_description, f, indent=4)

## Generating input file and output for testing

In [ ]:
time_steps = 100
u_dim = model.u_dim
x_latent_dim = model.x_latent_dim

# create and save input history
inputs = np.ones((time_steps, u_dim)) * np.arange(time_steps)[:, None]
columns = [f"u_0_{i}" for i in range(u_dim)]
index = pd.Index(np.arange(time_steps), name='time')
pd.DataFrame(data=inputs, columns=columns, index=index).to_csv("input.csv")

# buffers for outputs and states
results_y = torch.empty((time_steps, model.y_dim), dtype=torch.float32)
results_x = torch.empty((time_steps, x_latent_dim), dtype=torch.float32)

# initial state
x_prev = torch.zeros(x_latent_dim, dtype=torch.float32) # as in model description

for i in range(time_steps):
    u_row = torch.tensor(inputs[i], dtype=torch.float32)
    ynext, xnext = model(u_row, x_prev)    # new model: (u, x) -> (ynext, xnext)
    results_y[i] = ynext.detach().reshape(-1)
    results_x[i] = xnext.detach().reshape(-1)
    x_prev = xnext.detach()

# output into one CSV
output = pd.DataFrame(
    data=results_y.numpy(),
    columns=[f"ynext_{i}" for i in range(model.y_dim)],
    index=index
)
output.to_csv("output.csv")
# states into separate CSV
states = pd.DataFrame(
    data=results_x.numpy(),
    columns=[f"x{i}" for i in range(x_latent_dim)],
    index=index
)
states.to_csv("states.csv")
